In [1]:
try:
  import google.colab
  IN_COLAB = True
except:
  IN_COLAB = False

In [2]:
if IN_COLAB:
  # Install dependencies
  ! pip install --upgrade pip
  ! pip install czitools[ndv]
  ! pip install ipyfilechooser

In [3]:
# import the required libraries
from czitools.metadata_tools import czi_metadata
from czitools.utils import misc
from czitools.read_tools import read_tools
#from czitools.utils.ndv_tools import create_luts_ndv
from pathlib import Path
from ipyfilechooser import FileChooser
from IPython.display import display, HTML
import os
import dask.array as da
import requests
import glob
import ipywidgets as widgets
import ndv
import xarray as xr
#from typing import Any, cast

In [4]:
# try to find the folder with data and download otherwise from GitHub.

# Folder containing the input data
if IN_COLAB:
    INPUT_FOLDER = 'data/'
if not IN_COLAB:
    INPUT_FOLDER = '../../data/'

# Path to the data on GitHub
GITHUB_IMAGES_PATH = "https://raw.githubusercontent.com/sebi06/czitools/main/data.zip"

# Download data
if not (os.path.isdir(INPUT_FOLDER)):
    compressed_data = './data.zip'
    if not os.path.isfile(compressed_data):
        import io
        response = requests.get(GITHUB_IMAGES_PATH, stream=True)
        compressed_data = io.BytesIO(response.content)

    import zipfile
    with zipfile.ZipFile(compressed_data, 'r') as zip_accessor:
        zip_accessor.extractall('./')

In [5]:
if not IN_COLAB:
    # choose local file
    fc = FileChooser()
    fc.default_path = INPUT_FOLDER
    fc.filter_pattern = '*.czi'
    display(fc)

elif IN_COLAB:
    # list files inside the folder on gdrive
    czifiles = glob.glob(os.path.join(INPUT_FOLDER, "*.czi"))
    wd = widgets.Select(
        options=czifiles,
        description='CZI Files:',
        layout={'width': 'max-content'}
    )
    display(wd)

FileChooser(path='F:\GitHub\czitools\data', filename='', title='', show_hidden=False, select_desc='Select', ch…

In [6]:
if not IN_COLAB:
    filepath = fc.selected
elif IN_COLAB:
    filepath = wd.value

print(f"Selected File: {filepath}")

Selected File: F:\GitHub\czitools\data\CellDivision_T10_Z15_CH2_DCV_small.czi


In [7]:
# get the complete metadata at once as one big class
mdata = czi_metadata.CziMetadata(filepath)

# get the CZI metadata dictionary directly from filename
mdict = czi_metadata.create_md_dict_red(mdata, sort=False, remove_none=True)

# convert metadata dictionary to a pandas dataframe
mdframe = misc.md2dataframe(mdict)

# create a ipywdiget to show the dataframe with the metadata
wd = widgets.Output(layout={"scrollY": "auto", "height": "300px"})

with wd:
    display(HTML(mdframe.to_html()))
display(widgets.VBox(children=[wd]))

2026-03-12 11:33:13,020 - czitools - INFO - CZI dimensions found: ['T', 'C', 'Z']


100% |###################################################| Time:  0:00:00 1 of 1


In [8]:
result, dims, num_stacks, mdata = read_tools.read_stacks(
        filepath,
        zoom=1.0,
        use_dask=True,  # use lazy dask arrays (data read only when accessed)
        use_xarray=True,  # return xarray DataArray with labeled dimensions (STCZYX(A))
        stack_scenes=True,  # stack scenes with the same shape into one array (if False, return a list of arrays)
        adapt_metadata=True,  # adapt metadata according to the planes specified (SizeS, SizeT, SizeC, SizeZ)
    )

print("\n=== Results ===")
print(f"Array Shape: {result.shape}")
print(f"Number of stacks: {num_stacks}")
print(f"Dimension order: {dims}")

if isinstance(result, list):
    # List of per-scene arrays
    for idx, arr in enumerate(result):
        if isinstance(arr, xr.DataArray):
            print(f"Stack {idx}: dims={arr.dims}, shape={arr.shape}, dtype={arr.dtype}")
        else:
            print(f"Stack {idx}: shape={arr.shape}, dims={dims}, dtype={arr.dtype}")
else:
    # Single stacked array
    if isinstance(result, xr.DataArray):
        print(f"Stacked: dims={result.dims}, shape={result.shape}, dtype={result.dtype}")
    else:
        print(f"Stacked: shape={result.shape}, dims={dims}, dtype={result.dtype}")

    # With use_dask=True, result is backed by dask - no data loaded yet
    print(f"\nArray shape (no data loaded): {result.shape}")

2026-03-12 11:33:18,110 - czitools - INFO - CZI dimensions found: ['T', 'C', 'Z']


100% |###################################################| Time:  0:00:00 1 of 1


2026-03-12 11:33:18,192 - czitools - INFO - read_stacks: num_stacks=1 (selected from total=1), total_bounding_box={'T': (0, 10), 'Z': (0, 15), 'C': (0, 2), 'B': (0, 1), 'X': (0, 256), 'Y': (0, 256)}
2026-03-12 11:33:18,193 - czitools - INFO - read_stacks: canonical_dims=['T', 'C', 'Z'], dim_sizes={'T': 10, 'C': 2, 'Z': 15, 'S': 1}
2026-03-12 11:33:18,193 - czitools - INFO - read_stacks: Using lazy dask arrays - data will be read on demand
2026-03-12 11:33:18,360 - czitools - INFO - read_stacks: Stacking 1 stacks (all shapes equal: (10, 2, 15, 256, 256))

=== Results ===
Array Shape: (1, 10, 2, 15, 256, 256)
Number of stacks: 1
Dimension order: ['S', 'T', 'C', 'Z', 'Y', 'X']
Stacked: dims=('S', 'T', 'C', 'Z', 'Y', 'X'), shape=(1, 10, 2, 15, 256, 256), dtype=uint16

Array shape (no data loaded): (1, 10, 2, 15, 256, 256)


In [9]:
# verify whether result/results is a Dask array and display its structure
candidate = globals().get("results", globals().get("result"))

if candidate is None:
    print("No variable named 'result' or 'results' found yet.")
    print("Run the read_stacks cell first, then run this cell again.")
elif isinstance(candidate, xr.DataArray):
    is_dask = isinstance(candidate.data, da.Array)
    print(f"xarray.DataArray detected: {type(candidate)}")
    print(f"Backed by Dask: {is_dask}")
    if is_dask:
        darr = candidate.data
        print("\nDask-backed structure:")
        print(darr)
        print(f"shape   : {darr.shape}")
        print(f"dtype   : {darr.dtype}")
        print(f"chunks  : {darr.chunks}")
        print(f"numblocks: {darr.numblocks}")
        print(f"npartitions: {darr.npartitions}")
    else:
        print(f"shape: {candidate.shape}, dtype: {candidate.dtype}")
elif isinstance(candidate, da.Array):
    print("Dask array detected")
    print(candidate)
    print(f"shape   : {candidate.shape}")
    print(f"dtype   : {candidate.dtype}")
    print(f"chunks  : {candidate.chunks}")
    print(f"numblocks: {candidate.numblocks}")
    print(f"npartitions: {candidate.npartitions}")
else:
    print(f"Not a Dask array. Type: {type(candidate)}")
    if hasattr(candidate, "shape") and hasattr(candidate, "dtype"):
        print(f"shape: {candidate.shape}, dtype: {candidate.dtype}")

xarray.DataArray detected: <class 'xarray.core.dataarray.DataArray'>
Backed by Dask: True

Dask-backed structure:
dask.array<getitem, shape=(1, 10, 2, 15, 256, 256), dtype=uint16, chunksize=(1, 1, 1, 1, 256, 256), chunktype=numpy.ndarray>
shape   : (1, 10, 2, 15, 256, 256)
dtype   : uint16
chunks  : ((1,), (1, 1, 1, 1, 1, 1, 1, 1, 1, 1), (1, 1), (1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1), (256,), (256,))
numblocks: (1, 10, 2, 15, 1, 1)
npartitions: 300


In [10]:
# visualize array structure
result

<xarray.DataArray 'stack-d09fa843df2c40e3e701d6ca50c0d634' (S: 1, T: 10, C: 2,
                                                            Z: 15, Y: 256,
                                                            X: 256)> Size: 39MB
dask.array<getitem, shape=(1, 10, 2, 15, 256, 256), dtype=uint16, chunksize=(1, 1, 1, 1, 256, 256), chunktype=numpy.ndarray>
Coordinates:
  * S        (S) int64 8B 0
  * T        (T) int64 80B 0 1 2 3 4 5 6 7 8 9
  * C        (C) <U6 48B 'LED555' 'LED470'
  * Z        (Z) float64 120B 0.0 0.32 0.64 0.96 1.28 ... 3.52 3.84 4.16 4.48
  * Y        (Y) float64 2kB 0.0 0.091 0.182 0.273 ... 22.93 23.02 23.11 23.2
  * X        (X) float64 2kB 0.0 0.091 0.182 0.273 ... 22.93 23.02 23.11 23.2
Attributes:
    stack:          0
    filepath:       F:\GitHub\czitools\data\CellDivision_T10_Z15_CH2_DCV_smal...
    axes:           TCZYX
    subset_planes:  {'S': (0, 0), 'T': (0, 9), 'C': (0, 1), 'Z': (0, 14)}

In [11]:
# NOTE: NDV Jupyter backend currently expects slider coordinates as `range` objects.
# Passing an xarray DataArray directly can fail when coordinates are not ranges.

# Use raw array backing data to avoid xarray coordinate handling in NDV Jupyter view.
viewer_data = result.data if hasattr(result, "data") else result

# Display the data in NDV with scale metadata.
viewer = ndv.imshow(viewer_data, channel_mode="composite", channel_axis=2)

RFBOutputContext()

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>